In [1]:
!pip -q install faiss-cpu sentence-transformers
import os, json, time, shutil, itertools, numpy as np, faiss, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from collections import Counter
from google.colab import drive, files
drive.mount("/content/drive")

REPO = "/content/rag-image-captioning"
os.system(f"rm -rf {REPO} && git clone -q https://github.com/rafiebazine/rag-image-captioning.git {REPO} && cd {REPO} && git checkout -q hadil")
SRC  = "/content/drive/MyDrive/rag_embeddings"
PROC = f"{REPO}/phase_02_preprocessing/processed"
ORIG = f"{REPO}/phase_01_data_exploration/splitted_data_set"
OUT  = "/content/drive/MyDrive/coco2017_rag_preprocessing"
for d in ["splits", "metadata", "embeddings", "vector_db", "audit_eda/audit_logs", "audit_eda/figures"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
names = ["kb_10k", "val_100", "test_100", "test_1000"]

# ---------- 1) splits:
splits = {}
for n in names:
    orig = json.load(open(f"{ORIG}/{n}.json"))
    cl = {x["image_id"]: x for x in json.load(open(f"{PROC}/{n}_clean.json"))}
    for x in orig:
        x["raw_captions"] = cl[x["image_id"]]["captions_raw"]
        x["cleaned_captions"] = cl[x["image_id"]]["captions_clean"]
    json.dump(orig, open(f"{OUT}/splits/{n}.json", "w"))
    splits[n] = orig
kb = splits["kb_10k"]; by_id = {x["image_id"]: x for x in kb}

# ---------- 2) metadata:
def rel(a, b):
    ax, ay, aw, ah = a["bbox"]; bx, by, bw, bh = b["bbox"]
    if ax < bx + bw and bx < ax + aw and ay < by + bh and by < ay + ah: return "overlapping"
    dx, dy = (ax + aw/2) - (bx + bw/2), (ay + ah/2) - (by + bh/2)
    return ("left_of" if dx < 0 else "right_of") if abs(dx) > abs(dy) else ("above" if dy < 0 else "below")

meta_all = {}
for n in names:
    ms = json.load(open(f"{PROC}/metadata_{n}.json"))
    for m in ms:
        top = sorted(m["objects"], key=lambda o: -o["area"])[:5]
        m["attributes"] = []
        m["relations"] = [{"subject": a["category"], "relation": rel(a, b), "object": b["category"]}
                          for a, b in itertools.combinations(top, 2)]
    meta_all[n] = ms
json.dump(meta_all, open(f"{OUT}/metadata/visual_metadata.json", "w"))
json.dump({"detector": None,
           "source": "COCO 2017 ground-truth instance annotations (instances_train2017.json / instances_val2017.json)",
           "confidence_threshold": "N/A (ground truth)", "iou_cutoff": "N/A",
           "attributes": "not available in COCO instances (empty list)",
           "relations": "bbox-derived (overlapping/left_of/right_of/above/below) between the 5 largest objects per image",
           "note": "For val/test images this metadata is evaluation-only (oracle); never used as a retrieval query."},
          open(f"{OUT}/metadata/detector_config.json", "w"), indent=1)
meta_kb = {m["image_id"]: m for m in meta_all["kb_10k"]}

# ---------- 3) embeddings ----------
E = f"{OUT}/embeddings"
for n in names:
    shutil.copy(f"{SRC}/{n}_image_emb.npy", f"{E}/{n}_image_embeddings.npy")
shutil.copy(f"{SRC}/kb_text_emb.npy", f"{E}/kb_10k_text_embeddings_clip.npy")      # extra: CLIP text (image->text)
owners = json.load(open(f"{SRC}/kb_text_owners.json"))                              # row i -> [image_id, caption_idx]
ids = json.load(open(f"{SRC}/kb_10k_image_ids.json"))

from sentence_transformers import SentenceTransformer
dev = "cuda" if torch.cuda.is_available() else "cpu"
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=dev)
texts = [by_id[i]["cleaned_captions"][j] for i, j in owners]
t0 = time.time()
txt384 = sbert.encode(texts, batch_size=256, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype("float32")
t_sbert = round(time.time() - t0, 1)
assert txt384.shape == (50000, 384) and not np.isnan(txt384).any()
np.save(f"{E}/kb_10k_text_embeddings.npy", txt384)

img_emb = np.load(f"{E}/kb_10k_image_embeddings.npy")
clip_txt = np.load(f"{E}/kb_10k_text_embeddings_clip.npy")
json.dump({"image_encoder": {"checkpoint": "openai/clip-vit-base-patch32", "dim": 512, "batch_size": 128,
                             "normalisation": "L2", "dtype": "float32",
                             "preprocessing": "EXIF transpose, RGB, CLIPProcessor resize/normalise"},
           "text_encoder": {"checkpoint": "sentence-transformers/all-MiniLM-L6-v2", "dim": 384, "batch_size": 256,
                            "normalisation": "L2", "dtype": "float32", "source": "cleaned_captions (KB only)",
                            "encode_time_sec": t_sbert, "device": dev},
           "text_encoder_extra_clip": {"checkpoint": "openai/clip-vit-base-patch32", "dim": 512, "normalisation": "L2",
                                       "file": "kb_10k_text_embeddings_clip.npy", "use": "image->text retrieval (shared CLIP space)"},
           "shapes": {"kb_10k_image": list(img_emb.shape), "kb_10k_text": list(txt384.shape), "kb_10k_text_clip": list(clip_txt.shape)}},
          open(f"{E}/embedding_manifest.json", "w"), indent=1)

# ---------- 4) vector_db ----------
V = f"{OUT}/vector_db"
t0 = time.time(); idx_img = faiss.IndexFlatIP(512); idx_img.add(img_emb); t_i = time.time() - t0
t0 = time.time(); idx_txt = faiss.IndexFlatIP(384); idx_txt.add(txt384); t_t = time.time() - t0
idx_clip = faiss.IndexFlatIP(512); idx_clip.add(clip_txt)
faiss.write_index(idx_img, f"{V}/kb_10k_faiss.index")
faiss.write_index(idx_txt, f"{V}/kb_10k_text_faiss.index")
faiss.write_index(idx_clip, f"{V}/kb_10k_text_clip_faiss.index")
json.dump({"image_index": [{"vector_id": i, "image_id": iid, "file_name": by_id[iid]["file_name"],
                            "object_counts": meta_kb[iid]["object_counts"]} for i, iid in enumerate(ids)],
           "text_index": [{"vector_id": r, "image_id": iid, "caption_idx": j,
                           "raw_caption": by_id[iid]["raw_captions"][j].strip(),
                           "cleaned_caption": by_id[iid]["cleaned_captions"][j]} for r, (iid, j) in enumerate(owners)]},
          open(f"{V}/metadata_lookup.json", "w"))

def latency(index, q, k=10):
    ts = []
    for v in q:
        t = time.perf_counter(); index.search(v[None], k); ts.append((time.perf_counter() - t) * 1000)
    return round(float(np.mean(ts)), 3)
val_q = np.load(f"{E}/val_100_image_embeddings.npy")
json.dump({"library": f"faiss {faiss.__version__}", "index_type": "IndexFlatIP (exact)",
           "metric": "inner product on L2-normalised vectors (= cosine)",
           "indexes": {"kb_10k_faiss.index": {"vectors": idx_img.ntotal, "dim": 512, "content": "CLIP image embeddings"},
                       "kb_10k_text_faiss.index": {"vectors": idx_txt.ntotal, "dim": 384, "content": "SBERT caption embeddings"},
                       "kb_10k_text_clip_faiss.index": {"vectors": idx_clip.ntotal, "dim": 512, "content": "CLIP text embeddings"}},
           "size_mb": {f: round(os.path.getsize(f"{V}/{f}") / 1e6, 1) for f in os.listdir(V) if f.endswith(".index")},
           "indexing_time_sec": {"image": round(t_i, 3), "text_sbert": round(t_t, 3)},
           "latency_ms_per_query": {"image_to_image": latency(idx_img, val_q), "image_to_text_clip": latency(idx_clip, val_q)}},
          open(f"{V}/index_config.json", "w"), indent=1)

# ---------- 5) audit_logs ----------
A = f"{OUT}/audit_eda"
ids_s = {n: json.load(open(f"{SRC}/{n}_image_ids.json")) for n in names}
emb_s = {n: np.load(f"{E}/{n}_image_embeddings.npy") for n in names}
integrity = {"split_sizes": {n: len(ids_s[n]) for n in names},
             "image_embedding_shapes": {n: list(emb_s[n].shape) for n in names},
             "shape_alignment_ok": all(emb_s[n].shape == (len(ids_s[n]), 512) for n in names),
             "nan_found": {n: bool(np.isnan(emb_s[n]).any()) for n in names} | {"kb_text_sbert": bool(np.isnan(txt384).any()), "kb_text_clip": bool(np.isnan(clip_txt).any())},
             "duplicate_ids": {n: len(ids_s[n]) - len(set(ids_s[n])) for n in names},
             "five_captions_per_image": all(len(x["raw_captions"]) == 5 for n in names for x in splits[n]),
             "caption_mismatches_raw_vs_original": sum(x["captions"] != x["raw_captions"] for n in names for x in splits[n]),
             "kb_images_without_objects": sum(m["num_objects"] == 0 for m in meta_all["kb_10k"])}
assert integrity["shape_alignment_ok"] and not any(integrity["nan_found"].values()) and integrity["caption_mismatches_raw_vs_original"] == 0
json.dump(integrity, open(f"{A}/audit_logs/integrity_audit.json", "w"), indent=1)

sets = {n: set(ids_s[n]) for n in names}
log = [f"overlap {a} & {b}: {len(sets[a] & sets[b])}" for a, b in itertools.combinations(names, 2)]
eval_ids = sets["val_100"] | sets["test_100"] | sets["test_1000"]
log.append(f"set(indexed_db_ids).isdisjoint(eval_ids): {set(ids).isdisjoint(eval_ids)}")
log.append("rule: test image never retrieved for itself -> guaranteed, eval images are not in the index")
assert set(ids).isdisjoint(eval_ids)
open(f"{A}/audit_logs/leakage_audit.log", "w").write("\n".join(log) + "\n"); print("\n".join(log))

# dry-run retrieval
ex, top1 = [], []
for n in ["val_100", "test_100", "test_1000"]:
    s, i = idx_img.search(emb_s[n], 10)
    assert not ({ids[j] for row in i for j in row} & sets[n]); top1 += s[:, 0].tolist()
for n in ["val_100", "test_100"]:
    s, i = idx_img.search(emb_s[n][:3], 5); st, it = idx_clip.search(emb_s[n][:3], 3)
    for q in range(3):
        ex.append({"split": n, "query_image_id": ids_s[n][q],
                   "top5_images": [{"image_id": ids[j], "score": round(float(s[q, r]), 4)} for r, j in enumerate(i[q])],
                   "top3_captions": [{"image_id": owners[j][0], "caption": by_id[owners[j][0]]["raw_captions"][owners[j][1]].strip(),
                                      "score": round(float(st[q, r]), 4)} for r, j in enumerate(it[q])]})
json.dump({"status": "OK", "eval_image_ever_retrieved": False, "examples": ex}, open(f"{A}/audit_logs/dryrun_validation.json", "w"), indent=1)

# ---------- 6) figures 01 -> 10 ----------
F = f"{A}/figures"
def grab(src_name, dst):
    for p in [f"{PROC}/{src_name}", f"{PROC}/qc/{src_name}"]:
        if os.path.exists(p): shutil.copy(p, f"{F}/{dst}"); return
    print("missing:", src_name)
for s_, d_ in [("top50_objects.png", "01_top50_objects.png"), ("objects_per_image.png", "02_objects_per_image.png"),
               ("spatial_relations.png", "03_spatial_relations.png"), ("caption_length.png", "04_caption_length.png"),
               ("pos_distribution.png", "06_pos_distribution.png"), ("diversity_analysis.png", "07_diversity_analysis.png"),
               ("cooccurrence.png", "10_cooccurrence_heatmap.png")]:
    grab(s_, d_)

stop = set("a an the of on in with and is are to at near next by for from it its his her their this that there while up down out as be has have".split())
top = Counter(w for x in kb for c in x["cleaned_captions"] for w in c.split() if w not in stop).most_common(30)
plt.figure(figsize=(12, 4)); plt.bar([w for w, _ in top], [v for _, v in top]); plt.xticks(rotation=70)
plt.title("Top-30 content words (KB cleaned captions)"); plt.tight_layout(); plt.savefig(f"{F}/05_vocab_top_words.png", dpi=150); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(names, [np.mean([m["num_objects"] for m in meta_all[n]]) for n in names]); ax[0].set_title("Avg objects / image")
ax[1].bar(names, [100 * np.mean([m["num_objects"] == 0 for m in meta_all[n]]) for n in names]); ax[1].set_title("% images without objects")
plt.tight_layout(); plt.savefig(f"{F}/08_split_comparison.png", dpi=150); plt.close()

plt.figure(figsize=(7, 4)); plt.hist(top1, bins=40); plt.title("Top-1 image->image cosine (val/test queries)")
plt.savefig(f"{F}/09_retrieval_similarity.png", dpi=150); plt.close()

os.system(f'cd {OUT} && zip -qr /content/preproc_light.zip . -x "*.npy" "*.index"')
for root, _, fs in os.walk(OUT):
    for f in sorted(fs): print(os.path.join(root, f).replace(OUT + "/", ""), round(os.path.getsize(os.path.join(root, f)) / 1e6, 1), "MB")
files.download("/content/preproc_light.zip")

Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (8 outputs left out)

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

overlap kb_10k & val_100: 0
overlap kb_10k & test_100: 0
overlap kb_10k & test_1000: 0
overlap val_100 & test_100: 0
overlap val_100 & test_1000: 0
overlap test_100 & test_1000: 0
set(indexed_db_ids).isdisjoint(eval_ids): True
rule: test image never retrieved for itself -> guaranteed, eval images are not in the index
splits/kb_10k.json 9.8 MB
splits/test_100.json 0.1 MB
splits/test_1000.json 1.1 MB
splits/val_100.json 0.1 MB
metadata/detector_config.json 0.0 MB
metadata/visual_metadata.json 16.3 MB
embeddings/embedding_manifest.json 0.0 MB
embeddings/kb_10k_image_embeddings.npy 20.5 MB
embeddings/kb_10k_text_embeddings.npy 76.8 MB
embeddings/kb_10k_text_embeddings_clip.npy 102.4 MB
embeddings/test_1000_image_embeddings.npy 2.0 MB
embeddings/test_100_image_embeddings.npy 0.2 MB
embeddings/val_100_image_embeddings.npy 0.2 MB
vector_db/index_config.json 0.0 MB
vector_db/kb_10k_faiss.index 20.5 MB
vector_db/kb_10k_text_clip_faiss.index 102.4 MB
vector_db/kb_10k_text_faiss.index 76.8 MB
vec

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>